# Phase 1 & 2: Technical Setup + Pilot Analysis

**Interpretability of Portuguese Language Models via Sparse Autoencoders**

---

**Phase 1** — Technical Setup
- Install dependencies (SAELens, TransformerLens)
- Load Gemma 2 2B + Gemma Scope SAE (layer 13, 16k features)
- Basic test

**Phase 2** — Pilot Analysis
- Process ~1M tokens PT + EN (Wikipedia)
- Compute the Language Specificity Index (LSI)
- Identify the top-10 PT-specific features
- Generate visualizations

**Prerequisites:**
- GPU with ≥16GB VRAM (Colab T4 works)
- [Accept the Gemma 2 license](https://huggingface.co/google/gemma-2-2b) on HuggingFace

In [ ]:
!pip install sae-lens transformer-lens datasets -q

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

if torch.cuda.is_available():
    device = 'cuda'
    gpu_name = torch.cuda.get_device_name()
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu_name} ({gpu_mem:.1f} GB)')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = 'mps'
    print('Usando Apple Silicon (MPS)')
else:
    device = 'cpu'
    print('No GPU detected. Processing will be very slow.')

print(f'Device: {device}')
print(f'PyTorch: {torch.__version__}')

In [ ]:
from huggingface_hub import login
login()

---
# Phase 1: Technical Setup

We load the Gemma 2 2B model and the pre-trained SAE (Gemma Scope) for layer 13 with 16k features.

In [ ]:
from sae_lens import SAE, HookedSAETransformer

LAYER = 13
SAE_RELEASE = "gemma-scope-2b-pt-res-canonical"
SAE_ID = f"layer_{LAYER}/width_16k/canonical"

print("Loading Gemma 2 2B...")
model = HookedSAETransformer.from_pretrained_no_processing(
    "gemma-2-2b",
    device=device,
    dtype=torch.float16,
)
print(f"Model: {model.cfg.model_name} | Layers: {model.cfg.n_layers} | d_model: {model.cfg.d_model}")

print()
print(f"Loading SAE: {SAE_ID}...")
sae, cfg_dict, sparsity = SAE.from_pretrained(
    release=SAE_RELEASE,
    sae_id=SAE_ID,
    device=device,
)

HOOK_NAME = f"blocks.{LAYER}.hook_resid_post"
print(f"SAE: {sae.cfg.d_sae} features | d_in: {sae.cfg.d_in} | hook: {HOOK_NAME}")

In [ ]:
hook_name = HOOK_NAME

textos_teste = [
    ("PT", "O Brasil é o maior país da América do Sul e tem uma população diversa."),
    ("PT", "As meninas correram para a escola porque estavam atrasadas."),
    ("EN", "Brazil is the largest country in South America with a diverse population."),
    ("EN", "The girls ran to school because they were late."),
]

print("=== Quick Test ===")
print()
for lang, texto in textos_teste:
    tokens = model.to_tokens(texto)
    with torch.no_grad():
        _, cache = model.run_with_cache(tokens, names_filter=lambda name: name == hook_name)
        acts = cache[hook_name]
        feat_acts = sae.encode(acts)
    l0 = (feat_acts > 0).float().sum(dim=-1).mean().item()
    print(f'[{lang}] {texto[:55]}')
    print(f'     mean L0: {l0:.1f} features/token')
    print()

print("Phase 1 complete! Model and SAE working.")

---
# Phase 2: Pilot Analysis

We process ~1M tokens in PT and ~1M in EN (Wikipedia) to compute the **Language Specificity Index (LSI)** of each feature.

**LSI = (freq_PT - freq_EN) / (freq_PT + freq_EN)**

- LSI ≈ +1 → feature is active almost only in PT
- LSI ≈ -1 → feature is active almost only in EN
- LSI ≈ 0 → cross-lingual feature

In [ ]:
N_TOKENS = 1_000_000
SEQ_LEN = 256
BATCH_SIZE = 8

n_seqs = N_TOKENS // SEQ_LEN
n_batches = n_seqs // BATCH_SIZE

print(f"Tokens por language: {N_TOKENS:,}")
print(f"Sequences: {n_seqs:,} ({SEQ_LEN} tokens each)")
print(f"Batches: {n_batches:,} (batch size {BATCH_SIZE})")

In [ ]:
from datasets import load_dataset

tokenizer = model.tokenizer

def collect_tokens(dataset, n_tokens, desc="Tokenizando"):
    """Collects and tokenizes texts until reaching n_tokens."""
    all_ids = []
    n_articles = 0
    for article in tqdm(dataset, desc=desc):
        ids = tokenizer.encode(article["text"], add_special_tokens=False)
        all_ids.extend(ids)
        n_articles += 1
        if len(all_ids) >= n_tokens:
            break
    all_ids = all_ids[:n_tokens]
    n_seqs = len(all_ids) // SEQ_LEN
    tokens = torch.tensor(all_ids[:n_seqs * SEQ_LEN], dtype=torch.long).reshape(n_seqs, SEQ_LEN)
    print(f"  {n_articles} articles -> {tokens.numel():,} tokens ({tokens.shape[0]} seqs x {SEQ_LEN})")
    return tokens

print("Loading Wikipedia PT...")
wiki_pt = load_dataset("wikimedia/wikipedia", "20231101.pt", split="train", streaming=True)
tokens_pt = collect_tokens(wiki_pt, N_TOKENS, desc="PT")

print()
print("Loading Wikipedia EN...")
wiki_en = load_dataset("wikimedia/wikipedia", "20231101.en", split="train", streaming=True)
tokens_en = collect_tokens(wiki_en, N_TOKENS, desc="EN")

In [ ]:
@torch.no_grad()
def compute_feature_stats(model, sae, tokens, batch_size=BATCH_SIZE, desc=""):
    """Processes tokens through the model and SAE, returning per-feature statistics."""
    n_features = sae.cfg.d_sae
    hook = HOOK_NAME
    counts = torch.zeros(n_features, device=device)
    sums = torch.zeros(n_features, device=device)
    maxvals = torch.zeros(n_features, device=device)
    total = 0

    n_batches = (len(tokens) + batch_size - 1) // batch_size
    for i in tqdm(range(0, len(tokens), batch_size), desc=desc, total=n_batches):
        batch = tokens[i:i+batch_size].to(device)
        _, cache = model.run_with_cache(batch, names_filter=lambda n: n == hook)
        acts = cache[hook]
        feat_acts = sae.encode(acts)

        active = feat_acts > 0
        counts += active.float().sum(dim=(0, 1))
        sums += feat_acts.sum(dim=(0, 1))
        maxvals = torch.max(maxvals, feat_acts.amax(dim=(0, 1)))
        total += batch.numel()

        del cache, acts, feat_acts, active
        if device == "cuda":
            torch.cuda.empty_cache()

    return {"counts": counts.cpu(), "sums": sums.cpu(), "max": maxvals.cpu(), "total_tokens": total}


print("Processando PT (Wikipedia)...")
stats_pt = compute_feature_stats(model, sae, tokens_pt, desc="PT")
print(f'  {stats_pt["total_tokens"]:,} tokens | {(stats_pt["counts"] > 0).sum().item():,} active features')

print()
print("Processando EN (Wikipedia)...")
stats_en = compute_feature_stats(model, sae, tokens_en, desc="EN")
print(f'  {stats_en["total_tokens"]:,} tokens | {(stats_en["counts"] > 0).sum().item():,} active features')

In [ ]:
freq_pt = stats_pt["counts"] / stats_pt["total_tokens"]
freq_en = stats_en["counts"] / stats_en["total_tokens"]

lsi = (freq_pt - freq_en) / (freq_pt + freq_en + 1e-10)

alive = (stats_pt["counts"] + stats_en["counts"]) > 0
MIN_ACTS = 100
active = (stats_pt["counts"] + stats_en["counts"]) >= MIN_ACTS

n_total = sae.cfg.d_sae
n_dead = (~alive).sum().item()
n_alive = alive.sum().item()
n_active = active.sum().item()

print("=" * 55)
print("GENERAL STATISTICS")
print("=" * 55)
print(f"Total features:            {n_total:,}")
print(f"Dead features:           {n_dead:,} ({n_dead/n_total*100:.1f}%)")
print(f"Alive features:            {n_alive:,} ({n_alive/n_total*100:.1f}%)")
print(f"Active features (>={MIN_ACTS}):   {n_active:,} ({n_active/n_total*100:.1f}%)")

lsi_a = lsi[active]
print()
print("LSI DISTRIBUTION (active features)")
print(f"  Mean:     {lsi_a.mean().item():.4f}")
print(f"  Mediana:  {lsi_a.median().item():.4f}")
print(f"  PT-predominantes (LSI > 0.3):   {(lsi_a > 0.3).sum().item()}")
print(f"  EN-predominantes (LSI < -0.3):  {(lsi_a < -0.3).sum().item()}")
print(f"  Cross-lingual (|LSI| < 0.1):    {(lsi_a.abs() < 0.1).sum().item()}")

lsi_ranked_pt = lsi.clone()
lsi_ranked_pt[~active] = -2
top_pt_vals, top_pt_idx = lsi_ranked_pt.topk(10)

lsi_ranked_en = lsi.clone()
lsi_ranked_en[~active] = 2
top_en_vals, top_en_idx = (-lsi_ranked_en).topk(10)

print()
print("=" * 75)
print("TOP-10 PT-SPECIFIC FEATURES")
print("=" * 75)
fmt = '{:<4} {:<10} {:<10} {:<12} {:<12} {}'
print(fmt.format('#', 'Feature', 'LSI', 'Freq PT', 'Freq EN', 'Max PT'))
print("-" * 75)
for i in range(10):
    idx = top_pt_idx[i].item()
    print(fmt.format(
        i+1, idx,
        f'{lsi[idx].item():+.4f}',
        f'{freq_pt[idx].item():.6f}',
        f'{freq_en[idx].item():.6f}',
        f'{stats_pt["max"][idx].item():.2f}'
    ))

print()
print("=" * 75)
print("TOP-10 EN-SPECIFIC FEATURES (control)")
print("=" * 75)
print(fmt.format('#', 'Feature', 'LSI', 'Freq PT', 'Freq EN', 'Max EN'))
print("-" * 75)
for i in range(10):
    idx = top_en_idx[i].item()
    print(fmt.format(
        i+1, idx,
        f'{lsi[idx].item():+.4f}',
        f'{freq_pt[idx].item():.6f}',
        f'{freq_en[idx].item():.6f}',
        f'{stats_en["max"][idx].item():.2f}'
    ))

In [ ]:
@torch.no_grad()
def find_top_examples(model, sae, tokens, feature_indices,
                      n_examples=10, batch_size=BATCH_SIZE, max_batches=200):
    """Finds the tokens with the highest activation for each feature."""
    hook = HOOK_NAME
    feat_list = feature_indices.tolist()
    results = {f: [] for f in feat_list}
    tok = model.tokenizer

    actual_batches = min(max_batches, (len(tokens) + batch_size - 1) // batch_size)
    for i in tqdm(range(0, actual_batches * batch_size, batch_size),
                  desc="Searching for examples", total=actual_batches):
        if i >= len(tokens):
            break
        batch = tokens[i:i+batch_size].to(device)
        _, cache = model.run_with_cache(batch, names_filter=lambda n: n == hook)
        acts = cache[hook]
        feat_acts = sae.encode(acts)

        for f_idx in feat_list:
            vals = feat_acts[:, :, f_idx]
            for b in range(batch.shape[0]):
                max_val, max_pos = vals[b].max(dim=0)
                if max_val.item() > 0:
                    pos = max_pos.item()
                    start = max(0, pos - 8)
                    end = min(batch.shape[1], pos + 12)
                    ctx = tok.decode(batch[b, start:end].tolist())
                    token_str = tok.decode([batch[b, pos].item()])
                    results[f_idx].append({
                        "act": max_val.item(),
                        "token": token_str,
                        "context": ctx,
                    })

        del cache, acts, feat_acts
        if device == "cuda":
            torch.cuda.empty_cache()

    for f in results:
        results[f].sort(key=lambda x: x["act"], reverse=True)
        results[f] = results[f][:n_examples]
    return results


print("Finding examples for top-10 PT-specific features...")
examples_pt = find_top_examples(model, sae, tokens_pt, top_pt_idx)

print()
print("=" * 80)
print("ACTIVATING EXAMPLES — TOP-10 PT-SPECIFIC FEATURES")
print("=" * 80)

for rank, f_idx in enumerate(top_pt_idx.tolist()):
    print()
    print("-" * 70)
    print(f'Feature {f_idx} | LSI = {lsi[f_idx].item():+.4f} | Rank #{rank+1}')
    print("-" * 70)
    exs = examples_pt.get(f_idx, [])
    if not exs:
        print("  (no examples)")
        continue
    for ex in exs[:5]:
        print(f'  [{ex["act"]:.2f}] ...{ex["context"]}...')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma de LSI
lsi_vals = lsi[active].numpy()
axes[0].hist(lsi_vals, bins=100, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(0, color='black', linestyle='--', linewidth=0.8)
axes[0].axvline(0.3, color='red', linestyle=':', linewidth=1, label='LSI = +/- 0.3')
axes[0].axvline(-0.3, color='red', linestyle=':', linewidth=1)
axes[0].set_xlabel('Language Specificity Index (LSI)')
axes[0].set_ylabel('Contagem de features')
axes[0].set_title(f'Distribuição de LSI ({n_active:,} features ativas)')
axes[0].legend()

# Scatter: freq PT vs EN
fp = freq_pt[active].numpy()
fe = freq_en[active].numpy()
colors = lsi[active].numpy()
sc = axes[1].scatter(fe, fp, c=colors, cmap='RdBu', s=5, alpha=0.5, vmin=-1, vmax=1)
diag_max = max(fp.max(), fe.max())
axes[1].plot([0, diag_max], [0, diag_max], 'k--', linewidth=0.8, alpha=0.5)
axes[1].set_xlabel('Frequência EN')
axes[1].set_ylabel('Frequência PT')
axes[1].set_title('Frequência de Ativação: PT vs EN')
plt.colorbar(sc, ax=axes[1], label='LSI')

plt.tight_layout()
plt.savefig('fig1_lsi_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# Heatmap of top PT features
fig, ax = plt.subplots(figsize=(8, 6))
data = np.column_stack([freq_pt[top_pt_idx].numpy(), freq_en[top_pt_idx].numpy()])
im = ax.imshow(data, aspect='auto', cmap='YlOrRd')
ax.set_xticks([0, 1])
ax.set_xticklabels(['Wikipedia PT', 'Wikipedia EN'])
ax.set_yticks(range(10))
labels = [f'F{idx} (LSI={lsi[idx].item():+.2f})' for idx in top_pt_idx.tolist()]
ax.set_yticklabels(labels)
ax.set_title('Top-10 Features PT-específicas: Frequência por Idioma')
for i in range(10):
    for j in range(2):
        color = 'white' if data[i, j] > data.max() * 0.6 else 'black'
        ax.text(j, i, f'{data[i, j]:.5f}', ha='center', va='center', fontsize=9, color=color)
plt.colorbar(im, label='Frequência de ativação')
plt.tight_layout()
plt.savefig('fig2_top_features_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Save results
results = {
    'lsi': lsi, 'freq_pt': freq_pt, 'freq_en': freq_en,
    'stats_pt': stats_pt, 'stats_en': stats_en,
    'top_pt_idx': top_pt_idx, 'top_en_idx': top_en_idx,
    'active_mask': active,
}
torch.save(results, 'pilot_results.pt')
print('Results saved to pilot_results.pt')
print('Figuras: fig1_lsi_distribution.png, fig2_top_features_heatmap.png')